In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
class PPOAgent:
    def __init__(self, model_name, reward_model, tokenizer):
        self.policy_model = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_model = reward_model
        self.tokenizer = tokenizer
        self.ref_model = AutoModelForCausalLM.from_pretrained(model_name)  # Frozen reference
        
        # Freeze reference model
        for param in self.ref_model.parameters():
            param.requires_grad = False
    
    def generate(self, prompts, max_length=128, temperature=1.0):
        self.policy_model.eval()
        responses = []
        with torch.no_grad():
            for prompt in prompts:
                input_ids = self.tokenizer.encode(prompt, return_tensors='pt')
                
                outputs = self.policy_model.generate(
                    input_ids,
                    max_length=max_length,
                    temperature=temperature,
                    do_sample=True,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                
                response = self.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
                responses.append(response)
        
        return responses
    
    def compute_rewards(self, prompts, responses):
        rewards = []
        
        for prompt, response in zip(prompts, responses):
            full_text = f"{prompt}{response}"
            input_ids = self.tokenizer.encode(full_text, return_tensors='pt')
            
            with torch.no_grad():
                reward = self.reward_model(input_ids)
                rewards.append(reward.item())
        
        return torch.tensor(rewards)
    
    
    def ppo_update(self, prompts, responses, old_log_probs, advantages, clip_epsilon=0.2):
        self.policy_model.train()
        
        # Compute new log probabilities
        new_log_probs = []
        for prompt, response in zip(prompts, responses):
            full_text = f"{prompt}{response}"
            input_ids = self.tokenizer.encode(full_text, return_tensors='pt')
            
            outputs = self.policy_model(input_ids, labels=input_ids)
            log_probs = F.log_softmax(outputs.logits, dim=-1)
            
            # Get log prob of generated sequence
            response_ids = self.tokenizer.encode(response, return_tensors='pt')
            response_log_prob = log_probs[0, :len(response_ids[0]), response_ids[0]].mean()
            new_log_probs.append(response_log_prob)
        
        new_log_probs = torch.stack(new_log_probs)
        
        # Compute ratio
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        # PPO clipped objective
        clipped_ratio = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon)
        policy_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
        
        # KL penalty (prevent policy from drifting too far)
        with torch.no_grad():
            ref_log_probs = []
            for prompt, response in zip(prompts, responses):
                full_text = f"{prompt}{response}"
                input_ids = self.tokenizer.encode(full_text, return_tensors='pt')
                outputs = self.ref_model(input_ids, labels=input_ids)
                log_probs = F.log_softmax(outputs.logits, dim=-1)
                response_ids = self.tokenizer.encode(response, return_tensors='pt')
                ref_log_prob = log_probs[0, :len(response_ids[0]), response_ids[0]].mean()
                ref_log_probs.append(ref_log_prob)
        
        ref_log_probs = torch.stack(ref_log_probs)
        kl_penalty = (new_log_probs - ref_log_probs).mean()
        
        # Total loss
        total_loss = policy_loss + 0.1 * kl_penalty
        return total_loss